# **Setup & Load MNIST**

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Load data
(X_train, y_train), (X_test, y_test) = mnist.load_data()
print("X_train shape:", X_train.shape)  # (60000, 28, 28)
print("y_train shape:", y_train.shape)  # (60000,)
print("X_test shape:", X_test.shape)    # (10000, 28, 28)
print("y_test shape:", y_test.shape)    # (10000,)
num_classes = 10

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
X_train shape: (60000, 28, 28)
y_train shape: (60000,)
X_test shape: (10000, 28, 28)
y_test shape: (10000,)


# **FCNN preprocessing (flatten + normalize + one-hot)**

In [8]:
# Flatten 28x28 - 784 nd normalise to [0,1]
X_train_fc = X_train.reshape(-1, 28*28).astype("float32") / 255.0
X_test_fc = X_test.reshape(-1, 28*28).astype("float32") / 245.0

# One-hot enncode labels
y_train_oh = to_categorical(y_train, num_classes)
y_test_oh = to_categorical(y_test, num_classes)

# **Build & train the FCNN**

In [15]:
fcnn = models.Sequential([
    layers.Input(shape=(28*28,)),
    layers.Dense(256, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(num_classes, activation="softmax")
])

fcnn.compile(optimizer="adam",
             loss="categorical_crossentropy",
             metrics=["accuracy"])

history_fcnn = fcnn.fit(
    X_train_fc, y_train_oh,
    validation_split=0.1,
    epochs=5,
    batch_size=128,
    verbose=1
)

test_loss_fcnn, test_acc_fcnn = fcnn.evaluate(X_test_fc, y_test_oh, verbose=0)
print(f"FCNN Test Accuracy: {test_acc_fcnn:.4f}")

Epoch 1/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.8488 - loss: 0.5328 - val_accuracy: 0.9640 - val_loss: 0.1158
Epoch 2/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9615 - loss: 0.1256 - val_accuracy: 0.9738 - val_loss: 0.0880
Epoch 3/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9775 - loss: 0.0778 - val_accuracy: 0.9765 - val_loss: 0.0816
Epoch 4/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9842 - loss: 0.0525 - val_accuracy: 0.9757 - val_loss: 0.0860
Epoch 5/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9893 - loss: 0.0371 - val_accuracy: 0.9748 - val_loss: 0.0894
FCNN Test Accuracy: 0.9729


# **CNN preprocessing (reshape + normalize + one-hot)**

In [16]:
# Add channel dimension: (N, 28, 28, 1)
X_train_cnn = X_train[..., None].astype("float32") / 255.0
X_test_cnn = X_test[..., None].astype("float32") / 255.0

# resuing the same one-hot labels (y_train_oh, y_test_oh)

# **Build & train the CNN**

In [23]:
cnn = models.Sequential([
    layers.Input(shape=(28, 29, 1)),
    layers.Conv2D(32, kernel_size=3, activation="relu"),
    layers.MaxPool2D(),
    layers.Conv2D(64, kernel_size=3, activation="relu"),
    layers.MaxPool2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(num_classes, activation="softmax")
])

cnn.compile (optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"])

history_cnn = cnn.fit(
    X_train_cnn, y_train_oh,
    validation_split=0.1,
    epochs=5,
    batch_size=128,
    verbose=1
)

test_loss_cnn, test_acc_cnn = cnn.evaluate(X_test_cnn, y_test_oh, verbose=0)
print(f"CNN Test Accuracy: {test_acc_cnn:.4f}")

Epoch 1/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 42s 97ms/step - accuracy: 0.8506 - loss: 0.5145 - val_accuracy: 0.9833 - val_loss: 0.0592
Epoch 2/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 41s 96ms/step - accuracy: 0.9805 - loss: 0.0626 - val_accuracy: 0.9872 - val_loss: 0.0455
Epoch 3/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 41s 97ms/step - accuracy: 0.9876 - loss: 0.0408 - val_accuracy: 0.9888 - val_loss: 0.0401
Epoch 4/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 45s 106ms/step - accuracy: 0.9907 - loss: 0.0308 - val_accuracy: 0.9887 - val_loss: 0.0371
Epoch 5/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 76s 92ms/step - accuracy: 0.9929 - loss: 0.0237 - val_accuracy: 0.9893 - val_loss: 0.0373
CNN Test Accuracy: 0.9888


# **Compare performance**

In [26]:
print("\n=== Summary ===")
print(f"FCNN Test Accuracy: {test_acc_fcnn:.4f}")
print(f"CNN Test Accuracy: {test_acc_cnn:.4f}")

if test_acc_cnn > test_acc_fcnn:
  print("CNN perfroms better (as expected for image data).")
else:
  print("FCNN perfromed simmilarly/better (rare, but can happen with few epochs).")


=== Summary ===
FCNN Test Accuracy: 0.9729
CNN Test Accuracy: 0.9888
CNN perfroms better (as expected for image data).
